# 02 - Cholesky Representation of Density Matrices

Interactive exploration of the Cholesky parameterization:
  $$\rho = \frac{LL^\dagger}{\text{Tr}(LL^\dagger)}$$

where $L$ is a lower-triangular complex matrix.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt

from src.representation.cholesky import (
    dm_to_cholesky, cholesky_to_dm, cholesky_dim,
    _cholesky_matrix_to_vector, _vector_to_cholesky_matrix
)
from src.representation.constraints import is_valid_dm, check_dm_constraints
from src.data.states import haar_random_pure, hilbert_schmidt_random
from src.evaluation.metrics import purity

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 4)

## 1. Round-Trip Verification

Verify that $\rho \to x \to \rho'$ preserves the state exactly.

In [ ]:
for n_qubits in [1, 2, 3]:
    d = 2**n_qubits
    print(f'\nn_qubits={n_qubits}, d={d}, Cholesky dim={d*d}')
    
    # Pure state
    rho_pure = haar_random_pure(n_qubits, seed=42)
    x_pure = dm_to_cholesky(rho_pure)
    rho_pure_recon = cholesky_to_dm(x_pure)
    err_pure = np.max(np.abs(rho_pure - rho_pure_recon))
    print(f'  Pure state round-trip error: {err_pure:.2e}')
    print(f'  Pure state purity preserved: {purity(rho_pure_recon):.8f}')
    
    # Mixed state
    rho_mixed = hilbert_schmidt_random(n_qubits, seed=42)
    x_mixed = dm_to_cholesky(rho_mixed)
    rho_mixed_recon = cholesky_to_dm(x_mixed)
    err_mixed = np.max(np.abs(rho_mixed - rho_mixed_recon))
    print(f'  Mixed state round-trip error: {err_mixed:.2e}')
    print(f'  Mixed state purity preserved: {purity(rho_mixed_recon):.8f}')

## 2. Cholesky Vector Structure

Visualize how the Cholesky vector encodes the density matrix.

In [ ]:
n_qubits = 2
d = 2**n_qubits

# Generate a pure state and its Cholesky vector
rho = haar_random_pure(n_qubits, seed=42)
x = dm_to_cholesky(rho)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

# Density matrix
axes[0, 0].imshow(np.real(rho), cmap='RdBu_r')
axes[0, 0].set_title('rho (real)')
axes[0, 1].imshow(np.imag(rho), cmap='RdBu_r')
axes[0, 1].set_title('rho (imag)')

# Cholesky matrix L
L = _vector_to_cholesky_matrix(x, d)
axes[0, 2].imshow(np.abs(L), cmap='Blues')
axes[0, 2].set_title('|L| (Cholesky matrix)')

# Reconstructed from L
M = L @ L.T.conj()
rho_from_L = M / np.trace(M)
axes[1, 0].imshow(np.real(rho_from_L), cmap='RdBu_r')
axes[1, 0].set_title('LL^dagger/Tr (real)')

# Difference
axes[1, 1].imshow(np.abs(rho - rho_from_L), cmap='Reds')
axes[1, 1].set_title('|rho - LL^dagger/Tr|')

# Cholesky vector (1D)
axes[1, 2].bar(range(len(x)), x)
axes[1, 2].set_title(f'Cholesky vector ({len(x)} dims)')
axes[1, 2].set_xlabel('Index')

plt.suptitle('Cholesky Representation Structure', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Regularization for Near-Pure States

Pure states have zero eigenvalues, making Cholesky numerically unstable.
We apply $\rho_\epsilon = (1-\epsilon)\rho + \epsilon I/d$.

In [ ]:
n_qubits = 3
rho = haar_random_pure(n_qubits, seed=42)

epsilons = [0, 1e-10, 1e-8, 1e-6, 1e-4, 1e-2]

print(f'Original purity: {purity(rho):.10f}')
print(f'Eigenvalues: {np.sort(np.linalg.eigvalsh(rho))[::-1][:5]}...')
print()
print(f'{"eps":>10s}  {"Round-trip Error":>18s}  {"Recon Purity":>14s}')
print('-' * 50)

for eps in epsilons:
    try:
        x = dm_to_cholesky(rho, eps=eps)
        rho_recon = cholesky_to_dm(x)
        err = np.max(np.abs(rho - rho_recon))
        pur = purity(rho_recon)
        print(f'{eps:10.0e}  {err:18.6e}  {pur:14.10f}')
    except Exception as e:
        print(f'{eps:10.0e}  FAILED: {str(e)[:40]}')

## 4. Geometry of Cholesky Space

How does adding Gaussian noise in Cholesky space affect the density matrix?

In [ ]:
n_qubits = 1
d = 2**n_qubits

# Start from maximally mixed state
rho_0 = np.eye(d, dtype=np.complex128) / d
x_0 = dm_to_cholesky(rho_0)

# Add increasing noise in Cholesky space and observe the resulting state
noise_levels = [0.0, 0.1, 0.5, 1.0, 2.0, 5.0]

fig, axes = plt.subplots(2, len(noise_levels), figsize=(16, 6))

for i, sigma in enumerate(noise_levels):
    np.random.seed(42)
    x_noisy = x_0 + sigma * np.random.randn(*x_0.shape)
    rho_noisy = cholesky_to_dm(x_noisy)
    
    axes[0, i].imshow(np.real(rho_noisy), cmap='RdBu_r', vmin=-0.5, vmax=0.5)
    axes[0, i].set_title(f'Real, sigma={sigma}')
    axes[1, i].imshow(np.imag(rho_noisy), cmap='RdBu_r', vmin=-0.5, vmax=0.5)
    axes[1, i].set_title(f'Imag, purity={purity(rho_noisy):.3f}')

plt.suptitle('Effect of Cholesky-Space Noise on Density Matrix', fontsize=14)
plt.tight_layout()
plt.show()

## Key Insights

- **Round-trip fidelity**: Essentially perfect (< 1e-14) with proper regularization
- **Regularization**: $\epsilon = 10^{-6}$ is sufficient for pure states up to 3 qubits
- **Noise geometry**: Gaussian noise in Cholesky space produces plausible density matrices
  (Hermitian, PSD, trace-1 guaranteed by construction)
- **Diagonal entries**: Must be real and non-negative (from Cholesky decomposition of PSD matrix)